In [ ]:
from pytorch_grad_cam import GradCAM, AblationCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from typing import List, Dict
import numpy as np
import cv2
import torch
from torch import nn
from torchvision.transforms import Compose, Normalize, ToTensor
import timm
import os
import time
from PIL import Image
from efficientnet_pytorch import EfficientNet

# ?? ???? ???
def preprocess_image(
    img: np.ndarray, mean=[
        0.5, 0.5, 0.5], std=[
            0.5, 0.5, 0.5]) -> torch.Tensor:
    preprocessing = Compose([
        ToTensor(),
        Normalize(mean=mean, std=std)
    ])
    return preprocessing(img.copy()).unsqueeze(0)

In [ ]:
model = EfficientNet.from_pretrained('efficientnet-b2')

model.load_state_dict(torch.load('weights/2Cycle_TO.pth'), strict=False)
model.to('cuda')

In [ ]:
layer = model._conv_head
layer

In [ ]:
rgb_img = cv2.imread('infe_images/plant/to1.jpg')
rgb_img = cv2.resize(rgb_img, (200, 200))
rgb_img = np.float32(rgb_img) / 255 #shape: (200, 200, 3)

input_tensor = preprocess_image(rgb_img,
                                mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])


In [ ]:

targets = [ClassifierOutputTarget(0)] #ca1 = 0 ca2 = 1 to1 = 0, to2 = 1

# cam ?? ? ??
cam = AblationCAM(model=model, target_layers=[layer])
cam.batch_size = 1
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
grayscale_cam = grayscale_cam[0, :]

In [ ]:
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
display(Image.fromarray(visualization))